In [ ]:
#@title **11B: Download All Models (ZIP)** { display-mode: "form" }
# Create a single ZIP with all models for easy download
import zipfile

zip_path = '/content/v5_all_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for model_name in ALL_MODELS:
        # YOLO models (best.pt)
        pt_path = f'{DRIVE_MODELS}/{model_name}_best.pt'
        if os.path.exists(pt_path):
            zf.write(pt_path, f'{model_name}/best.pt')
            continue
        
        # VideoMAE models (directory)
        model_dir = f'{DRIVE_MODELS}/{model_name}'
        if os.path.exists(model_dir):
            for root, dirs, files in os.walk(model_dir):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.join(model_name, os.path.relpath(full_path, model_dir))
                    zf.write(full_path, arcname)

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"All models ZIP: {size_mb:.1f}MB")
    
    try:
        colab_files.download(zip_path)
        print("Download triggered!")
    except:
        print(f"Download manually from Colab files panel: {zip_path}")
        print(f"Or from Drive: {DRIVE_MODELS}")
else:
    print("No models to zip. Train models first!")

print("\n=== V5 PIPELINE COMPLETE ===")

In [ ]:
#@title **11A: Final Report** { display-mode: "form" }
print("=" * 70)
print("  V5 TRAINING PIPELINE — FINAL REPORT")
print("=" * 70)

# Check all expected models
ALL_MODELS = [
    'videomae_basketball_v5', 'videomae_football_v5', 'videomae_lacrosse_v5',
    'yolo_outcome_basketball_v5', 'yolo_outcome_football_v5', 'yolo_outcome_lacrosse_v5',
    'jersey_ocr_v5', 'player_detector_v5', 'ball_detector_v5', 'referee_detector_v5'
]

found = 0
missing = 0
for model_name in ALL_MODELS:
    # Check Drive for model
    drive_path = f'{DRIVE_MODELS}/{model_name}_best.pt'
    drive_dir = f'{DRIVE_MODELS}/{model_name}'
    
    if os.path.exists(drive_path):
        size = os.path.getsize(drive_path) / 1024 / 1024
        print(f"  ✓ {model_name}: {size:.1f}MB")
        found += 1
    elif os.path.exists(drive_dir):
        print(f"  ✓ {model_name}: saved (HuggingFace format)")
        found += 1
    else:
        print(f"  ✗ {model_name}: NOT FOUND")
        missing += 1

print(f"\n  Models: {found}/{len(ALL_MODELS)} trained")

# Show API costs
if os.path.exists(DRIVE_COSTS):
    with open(DRIVE_COSTS) as f:
        costs = json.load(f)
    print(f"\n  API Costs:")
    for k, v in costs.items():
        print(f"    {k}: ${v:.4f}")

print(f"\n  All models saved to: {DRIVE_MODELS}")
print("=" * 70)

## Section 11: Final Report + Download All Models\nSummary of all trained models, validation metrics, and bulk download.

In [ ]:
#@title **10: Referee Detector v5** { display-mode: "form" }
REF_DATA_DIR = '/content/referee_detect_data'
os.makedirs(REF_DATA_DIR, exist_ok=True)

if not ROBOFLOW_API_KEY:
    print("ERROR: Set ROBOFLOW_API_KEY in Section 0!")
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    ref_datasets = [
        ("referee-detection-mstxn", 1),
        ("referee-detection-oijfj", 1),
        ("sports-referee-xdmth", 1),
    ]
    
    total_images = 0
    for ds_name, version in ref_datasets:
        for v in [version, 1, 2, 3]:
            try:
                project = rf.workspace().project(ds_name)
                dataset = project.version(v).download("yolov8", location=f'{REF_DATA_DIR}/{ds_name}')
                img_count = len(glob.glob(f'{REF_DATA_DIR}/{ds_name}/train/images/*'))
                total_images += img_count
                print(f"  {ds_name} v{v}: {img_count} images")
                break
            except:
                if v == 3:
                    print(f"  {ds_name}: Failed")
    
    if total_images > 0:
        MERGED_REF = '/content/referee_detect_merged'
        for split in ['train', 'val']:
            os.makedirs(f'{MERGED_REF}/{split}/images', exist_ok=True)
            os.makedirs(f'{MERGED_REF}/{split}/labels', exist_ok=True)
        
        for ds_dir in glob.glob(f'{REF_DATA_DIR}/*'):
            if not os.path.isdir(ds_dir):
                continue
            ds_name = os.path.basename(ds_dir)
            for split in ['train', 'valid', 'val', 'test']:
                img_dir = f'{ds_dir}/{split}/images'
                lbl_dir = f'{ds_dir}/{split}/labels'
                if not os.path.exists(img_dir):
                    continue
                target = 'val' if split in ['valid', 'val', 'test'] else 'train'
                for img in glob.glob(f'{img_dir}/*'):
                    fname = f'{ds_name}_{os.path.basename(img)}'
                    shutil.copy2(img, f'{MERGED_REF}/{target}/images/{fname}')
                    lbl = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img))[0] + '.txt')
                    if os.path.exists(lbl):
                        with open(lbl) as f:
                            lines = ['0 ' + ' '.join(l.strip().split()[1:]) for l in f if len(l.strip().split()) >= 5]
                        if lines:
                            with open(f'{MERGED_REF}/{target}/labels/{os.path.splitext(fname)[0]}.txt', 'w') as f:
                                f.write('\n'.join(lines) + '\n')
        
        data_yaml = {'path': MERGED_REF, 'train': 'train/images', 'val': 'val/images', 'nc': 1, 'names': {0: 'referee'}}
        with open(f'{MERGED_REF}/data.yaml', 'w') as f:
            yaml.dump(data_yaml, f)
        
        tc = len(glob.glob(f'{MERGED_REF}/train/images/*'))
        vc = len(glob.glob(f'{MERGED_REF}/val/images/*'))
        print(f"\nReferee detector dataset: {tc} train, {vc} val")
        
        model, results = safe_train(
            'yolov8m.pt',
            data=f'{MERGED_REF}/data.yaml',
            epochs=80,
            imgsz=640,
            batch=DEFAULT_BATCH,
            model_name='referee_detector_v5',
        )
        
        download_model('referee_detector_v5')
        print("\n=== Referee Detector v5 trained! ===")
    else:
        print("No referee datasets found — skipping.")

## Section 10: Referee Detector v5 (NEW)\nDetect referees to filter them out of player tracking.\nPrevents jersey OCR from reading referee numbers as player numbers.

In [ ]:
#@title **9: Ball Detector v5** { display-mode: "form" }
BALL_DATA_DIR = '/content/ball_detect_data'
os.makedirs(BALL_DATA_DIR, exist_ok=True)

if not ROBOFLOW_API_KEY:
    print("ERROR: Set ROBOFLOW_API_KEY in Section 0!")
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    ball_datasets = [
        ("basketball-detection-zirim", 1),     # Basketball
        ("football-detection-0oqwu", 1),       # Football
        ("ball-detection-rhlsr", 1),            # General ball
        ("lacrosse-ball-detection-xhfkp", 1),  # Lacrosse ball
    ]
    
    total_images = 0
    for ds_name, version in ball_datasets:
        for v in [version, 1, 2, 3]:
            try:
                project = rf.workspace().project(ds_name)
                dataset = project.version(v).download("yolov8", location=f'{BALL_DATA_DIR}/{ds_name}')
                img_count = len(glob.glob(f'{BALL_DATA_DIR}/{ds_name}/train/images/*'))
                total_images += img_count
                print(f"  {ds_name} v{v}: {img_count} images")
                break
            except:
                if v == 3:
                    print(f"  {ds_name}: Failed")
    
    if total_images > 0:
        # Merge all → single class "ball"
        MERGED_BALL = '/content/ball_detect_merged'
        for split in ['train', 'val']:
            os.makedirs(f'{MERGED_BALL}/{split}/images', exist_ok=True)
            os.makedirs(f'{MERGED_BALL}/{split}/labels', exist_ok=True)
        
        for ds_dir in glob.glob(f'{BALL_DATA_DIR}/*'):
            if not os.path.isdir(ds_dir):
                continue
            ds_name = os.path.basename(ds_dir)
            for split in ['train', 'valid', 'val', 'test']:
                img_dir = f'{ds_dir}/{split}/images'
                lbl_dir = f'{ds_dir}/{split}/labels'
                if not os.path.exists(img_dir):
                    continue
                target = 'val' if split in ['valid', 'val', 'test'] else 'train'
                for img in glob.glob(f'{img_dir}/*'):
                    fname = f'{ds_name}_{os.path.basename(img)}'
                    shutil.copy2(img, f'{MERGED_BALL}/{target}/images/{fname}')
                    lbl = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img))[0] + '.txt')
                    if os.path.exists(lbl):
                        with open(lbl) as f:
                            lines = ['0 ' + ' '.join(l.strip().split()[1:]) for l in f if len(l.strip().split()) >= 5]
                        if lines:
                            with open(f'{MERGED_BALL}/{target}/labels/{os.path.splitext(fname)[0]}.txt', 'w') as f:
                                f.write('\n'.join(lines) + '\n')
        
        data_yaml = {'path': MERGED_BALL, 'train': 'train/images', 'val': 'val/images', 'nc': 1, 'names': {0: 'ball'}}
        with open(f'{MERGED_BALL}/data.yaml', 'w') as f:
            yaml.dump(data_yaml, f)
        
        tc = len(glob.glob(f'{MERGED_BALL}/train/images/*'))
        vc = len(glob.glob(f'{MERGED_BALL}/val/images/*'))
        print(f"\nBall detector dataset: {tc} train, {vc} val")
        
        model, results = safe_train(
            'yolov8m.pt',
            data=f'{MERGED_BALL}/data.yaml',
            epochs=80,
            imgsz=640,
            batch=DEFAULT_BATCH,
            model_name='ball_detector_v5',
        )
        
        download_model('ball_detector_v5')
        print("\n=== Ball Detector v5 trained! ===")
    else:
        print("No ball datasets found — skipping. You can add datasets manually.")

## Section 9: Ball Detector v5 (NEW)\nDetect the ball in sports footage — critical for identifying active plays.\nKnowing ball position helps determine who has possession and what play is happening.

In [ ]:
#@title **8B: Merge Player Data + Train** { display-mode: "form" }
MERGED_PLAYER = '/content/player_detect_merged'
for split in ['train', 'val']:
    os.makedirs(f'{MERGED_PLAYER}/{split}/images', exist_ok=True)
    os.makedirs(f'{MERGED_PLAYER}/{split}/labels', exist_ok=True)

# Merge all player datasets, remap all classes to class 0 (player)
for ds_dir in glob.glob(f'{PLAYER_DATA_DIR}/*'):
    if not os.path.isdir(ds_dir):
        continue
    ds_name = os.path.basename(ds_dir)
    
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = f'{ds_dir}/{split}/images'
        lbl_dir = f'{ds_dir}/{split}/labels'
        if not os.path.exists(img_dir):
            continue
        
        target_split = 'val' if split in ['valid', 'val', 'test'] else 'train'
        
        for img in glob.glob(f'{img_dir}/*'):
            fname = f'{ds_name}_{os.path.basename(img)}'
            shutil.copy2(img, f'{MERGED_PLAYER}/{target_split}/images/{fname}')
            
            lbl = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img))[0] + '.txt')
            if os.path.exists(lbl):
                # Remap all classes to 0 (player)
                with open(lbl) as f:
                    lines = f.readlines()
                remapped = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        parts[0] = '0'  # All → player
                        remapped.append(' '.join(parts))
                if remapped:
                    out_lbl = f'{MERGED_PLAYER}/{target_split}/labels/{os.path.splitext(fname)[0]}.txt'
                    with open(out_lbl, 'w') as f:
                        f.write('\n'.join(remapped) + '\n')

# Create data.yaml
data_yaml = {
    'path': MERGED_PLAYER,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': {0: 'player'}
}
with open(f'{MERGED_PLAYER}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

train_c = len(glob.glob(f'{MERGED_PLAYER}/train/images/*'))
val_c = len(glob.glob(f'{MERGED_PLAYER}/val/images/*'))
print(f"Player detector dataset: {train_c} train, {val_c} val")

# Train
model, results = safe_train(
    'yolov8m.pt',
    data=f'{MERGED_PLAYER}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=DEFAULT_BATCH,
    model_name='player_detector_v5',
)

download_model('player_detector_v5')
print("\n=== Player Detector v5 trained! ===")

In [ ]:
#@title **8A: Download Player Detection Datasets** { display-mode: "form" }
PLAYER_DATA_DIR = '/content/player_detect_data'
os.makedirs(PLAYER_DATA_DIR, exist_ok=True)

if not ROBOFLOW_API_KEY:
    print("ERROR: Set ROBOFLOW_API_KEY in Section 0!")
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Multiple player/person detection datasets for robust training
    player_datasets = [
        ("football-players-detection-3zvbc", 1),  # Football players
        ("basketball-players-fy4c2", 1),           # Basketball players
        ("players-detection-mcsaz", 1),            # General sports players
        ("nfl-players-4down-s8gop", 1),            # NFL specific
    ]
    
    total_images = 0
    for ds_name, version in player_datasets:
        for v in [version, 1, 2, 3]:
            try:
                project = rf.workspace().project(ds_name)
                dataset = project.version(v).download("yolov8", location=f'{PLAYER_DATA_DIR}/{ds_name}')
                img_count = len(glob.glob(f'{PLAYER_DATA_DIR}/{ds_name}/train/images/*'))
                total_images += img_count
                print(f"  {ds_name} v{v}: {img_count} images")
                break
            except Exception as e:
                if v == 3:
                    print(f"  {ds_name}: All versions failed — {e}")
    
    print(f"\nTotal player detection images: {total_images}")
    
    if total_images > 0:
        drive_player = f'{DRIVE_DATA}/player_detect'
        shutil.copytree(PLAYER_DATA_DIR, drive_player, dirs_exist_ok=True)
        print(f"Backed up to Drive: {drive_player}")

## Section 8: Player Detector v5 (FIXED)\n**Critical fix:** Uses Roboflow person/player datasets directly instead of empty Claude-labeled data.\nDetects players in sports footage for jersey crop extraction.

In [ ]:
#@title **7D: Train Jersey OCR v5** { display-mode: "form" }
# CRITICAL: fliplr=0.0 prevents 6/9 confusion
model, results = safe_train(
    'yolov8m.pt',
    data=f'{MERGED_OCR}/data.yaml',
    epochs=120,
    imgsz=640,
    batch=DEFAULT_BATCH,
    model_name='jersey_ocr_v5',
    extra_args={
        'fliplr': 0.0,      # NO horizontal flip — prevents 6↔9 confusion
        'mosaic': 0.5,       # Reduced mosaic — digits need spatial context
        'degrees': 5.0,      # Slight rotation only
        'scale': 0.3,        # Moderate scale augmentation
    }
)

download_model('jersey_ocr_v5')
print("\n=== Jersey OCR v5 trained! ===")

In [ ]:
#@title **7C: Merge OCR Data + Create data.yaml** { display-mode: "form" }
import yaml

MERGED_OCR = '/content/jersey_ocr_merged'
for split in ['train', 'val']:
    os.makedirs(f'{MERGED_OCR}/{split}/images', exist_ok=True)
    os.makedirs(f'{MERGED_OCR}/{split}/labels', exist_ok=True)

# Copy Roboflow data
for ds_dir in glob.glob(f'{OCR_DATA_DIR}/*'):
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = f'{ds_dir}/{split}/images'
        lbl_dir = f'{ds_dir}/{split}/labels'
        if os.path.exists(img_dir):
            target_split = 'val' if split in ['valid', 'val', 'test'] else 'train'
            ds_name = os.path.basename(ds_dir)
            for img in glob.glob(f'{img_dir}/*'):
                fname = f'{ds_name}_{os.path.basename(img)}'
                shutil.copy2(img, f'{MERGED_OCR}/{target_split}/images/{fname}')
                lbl = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img))[0] + '.txt')
                if os.path.exists(lbl):
                    shutil.copy2(lbl, f'{MERGED_OCR}/{target_split}/labels/{os.path.splitext(fname)[0]}.txt')

# Copy synthetic data (80/20 split)
synth_images = sorted(glob.glob(f'{SYNTH_DIR}/images/*.jpg'))
split_idx = int(len(synth_images) * 0.8)
for i, img_path in enumerate(synth_images):
    fname = os.path.basename(img_path)
    lbl_fname = fname.replace('.jpg', '.txt')
    split = 'train' if i < split_idx else 'val'
    shutil.copy2(img_path, f'{MERGED_OCR}/{split}/images/{fname}')
    shutil.copy2(f'{SYNTH_DIR}/labels/{lbl_fname}', f'{MERGED_OCR}/{split}/labels/{lbl_fname}')

# Create data.yaml — 10 DIGIT CLASSES ONLY
data_yaml = {
    'path': MERGED_OCR,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 10,
    'names': {i: str(i) for i in range(10)}
}
with open(f'{MERGED_OCR}/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

train_count = len(glob.glob(f'{MERGED_OCR}/train/images/*'))
val_count = len(glob.glob(f'{MERGED_OCR}/val/images/*'))
print(f"Merged OCR dataset: {train_count} train, {val_count} val")
print(f"Classes: 10 digits (0-9) — NOT 100 numbers!")

In [ ]:
#@title **7B: Fix Class Mapping + Generate Synthetic Data** { display-mode: "form" }
from PIL import Image, ImageDraw, ImageFont
import random

SYNTH_DIR = '/content/jersey_ocr_synthetic'
os.makedirs(f'{SYNTH_DIR}/images', exist_ok=True)
os.makedirs(f'{SYNTH_DIR}/labels', exist_ok=True)

# Remap any Roboflow dataset classes to 10 digits (0-9)
def remap_labels_to_digits(dataset_dir):
    """Ensure all labels use class IDs 0-9 for digits 0-9."""
    label_dirs = glob.glob(f'{dataset_dir}/*/labels')
    remapped = 0
    
    for label_dir in label_dirs:
        data_yaml = os.path.join(os.path.dirname(label_dir), 'data.yaml')
        class_map = {}
        
        if os.path.exists(data_yaml):
            import yaml
            with open(data_yaml) as f:
                data = yaml.safe_load(f)
            names = data.get('names', [])
            # Map original class ID → digit value
            for orig_id, name in enumerate(names):
                name_clean = str(name).strip()
                if name_clean.isdigit() and 0 <= int(name_clean) <= 9:
                    class_map[orig_id] = int(name_clean)
        
        if not class_map:
            continue
        
        for label_file in glob.glob(f'{label_dir}/*.txt'):
            lines = []
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        orig_class = int(parts[0])
                        if orig_class in class_map:
                            parts[0] = str(class_map[orig_class])
                            lines.append(' '.join(parts))
            if lines:
                with open(label_file, 'w') as f:
                    f.write('\n'.join(lines) + '\n')
                remapped += 1
    
    print(f"  Remapped {remapped} label files to digit classes 0-9")

# Remap all downloaded datasets
for ds_dir in glob.glob(f'{OCR_DATA_DIR}/*'):
    if os.path.isdir(ds_dir):
        remap_labels_to_digits(ds_dir)

# Generate synthetic jersey digit images
JERSEY_COLORS = [(255,0,0), (0,0,255), (255,255,255), (0,128,0), 
                 (255,165,0), (128,0,128), (0,0,0), (255,255,0)]
TEXT_COLORS = [(255,255,255), (0,0,0), (255,255,0), (255,0,0)]

NUM_SYNTHETIC = 2000
print(f"\nGenerating {NUM_SYNTHETIC} synthetic jersey images...")

for i in range(NUM_SYNTHETIC):
    # Random jersey number (1-99)
    number = random.randint(0, 99)
    digits = str(number)
    
    # Create image
    w, h = 640, 640
    bg_color = random.choice(JERSEY_COLORS)
    img = Image.new('RGB', (w, h), bg_color)
    draw = ImageDraw.Draw(img)
    
    # Add noise/texture
    for _ in range(random.randint(50, 200)):
        x1 = random.randint(0, w-1)
        y1 = random.randint(0, h-1)
        noise_color = tuple(max(0, min(255, c + random.randint(-30, 30))) for c in bg_color)
        draw.rectangle([x1, y1, x1+random.randint(2,8), y1+random.randint(2,8)], fill=noise_color)
    
    # Draw digits with REALISTIC positions and sizes
    text_color = random.choice(TEXT_COLORS)
    font_size = random.randint(60, 150)
    
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", font_size)
    except:
        font = ImageFont.load_default()
    
    # Position: center area with some randomness
    total_width = len(digits) * (font_size * 0.7)
    start_x = (w - total_width) / 2 + random.randint(-50, 50)
    y_pos = h / 2 - font_size / 2 + random.randint(-80, 80)
    
    labels_lines = []
    for j, digit in enumerate(digits):
        x = start_x + j * (font_size * 0.7)
        draw.text((x, y_pos), digit, fill=text_color, font=font)
        
        # REALISTIC bbox per digit (NOT full-frame!)
        digit_w = font_size * 0.65
        digit_h = font_size * 1.1
        cx = (x + digit_w / 2) / w
        cy = (y_pos + digit_h / 2) / h
        bw = digit_w / w
        bh = digit_h / h
        
        # Clamp to valid range
        cx = max(0.01, min(0.99, cx))
        cy = max(0.01, min(0.99, cy))
        bw = max(0.02, min(0.5, bw))
        bh = max(0.02, min(0.5, bh))
        
        class_id = int(digit)
        labels_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    
    img.save(f'{SYNTH_DIR}/images/synth_{i:05d}.jpg')
    with open(f'{SYNTH_DIR}/labels/synth_{i:05d}.txt', 'w') as f:
        f.write('\n'.join(labels_lines) + '\n')

print(f"Generated {NUM_SYNTHETIC} synthetic images with proper digit bboxes")
print(f"Sample bbox sizes: ~{font_size*0.65/640:.3f}w x {font_size*1.1/640:.3f}h (NOT 0.9x0.9!)")

In [ ]:
#@title **7A: Download Jersey OCR Datasets** { display-mode: "form" }
from roboflow import Roboflow

OCR_DATA_DIR = '/content/jersey_ocr_data'
os.makedirs(OCR_DATA_DIR, exist_ok=True)

# DIGIT classes: 0-9 only (NOT 0-99 numbers)
DIGIT_CLASSES = [str(i) for i in range(10)]

if not ROBOFLOW_API_KEY:
    print("ERROR: Set ROBOFLOW_API_KEY in Section 0!")
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Download jersey number datasets with digit-level annotations
    datasets_to_try = [
        ("jersey-number-detection-qkeap", 2),  # Jersey numbers
        ("jersey-number-detection-1jlom", 1),  # Another jersey dataset
        ("jersey-number-qsdyg", 1),            # More jersey data
    ]
    
    total_images = 0
    for ds_name, version in datasets_to_try:
        try:
            project = rf.workspace().project(ds_name)
            dataset = project.version(version).download("yolov8", location=f'{OCR_DATA_DIR}/{ds_name}')
            
            img_count = len(glob.glob(f'{OCR_DATA_DIR}/{ds_name}/train/images/*'))
            total_images += img_count
            print(f"  {ds_name}: {img_count} images")
        except Exception as e:
            print(f"  {ds_name}: Failed — {e}")
            # Try alternative versions
            for alt_v in [1, 2, 3]:
                if alt_v == version:
                    continue
                try:
                    dataset = project.version(alt_v).download("yolov8", location=f'{OCR_DATA_DIR}/{ds_name}')
                    img_count = len(glob.glob(f'{OCR_DATA_DIR}/{ds_name}/train/images/*'))
                    total_images += img_count
                    print(f"  {ds_name} v{alt_v}: {img_count} images")
                    break
                except:
                    pass
    
    print(f"\nTotal jersey OCR images: {total_images}")
    
    # Backup to Drive
    drive_ocr = f'{DRIVE_DATA}/jersey_ocr'
    if total_images > 0:
        shutil.copytree(OCR_DATA_DIR, drive_ocr, dirs_exist_ok=True)
        print(f"Backed up to Drive: {drive_ocr}")

## Section 7: Jersey OCR v5 (FIXED)\n**Critical fixes from v4 audit:**\n- Uses 10 digit classes (0-9), NOT 100 number classes (0-99)\n- Left-to-right digit concatenation for multi-digit numbers\n- Realistic small bounding boxes per digit, NOT full-frame\n- `fliplr=0.0` to prevent 6/9 confusion\n- Roboflow jersey datasets for real training data

In [ ]:
#@title **6B: Train YOLO Outcome Classifiers** { display-mode: "form" }
for sport_name in ['basketball', 'football', 'lacrosse']:
    cls_dir = f'/content/yolo_cls/{sport_name}'
    if not os.path.exists(f'{cls_dir}/train'):
        print(f"{sport_name}: No data, skipping")
        continue
    
    model_name = f'yolo_outcome_{sport_name}_v5'
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    
    model, results = safe_train(
        'yolov8m-cls.pt',
        data=cls_dir,
        epochs=80,
        imgsz=640,
        batch=DEFAULT_BATCH,
        model_name=model_name,
    )
    
    download_model(model_name)
    print(f"{model_name} complete!")
    torch.cuda.empty_cache()

print("\n=== All outcome classifiers trained ===")

In [ ]:
#@title **6A: Prepare YOLO Classification Data** { display-mode: "form" }
# Extract middle frames from labeled clips → classification dataset
# Structure: /content/yolo_cls/{sport}/train/{class}/{image}.jpg

for sport_name in ['basketball', 'football', 'lacrosse']:
    labels_file = f'{DRIVE_DATA}/labels/{sport_name}_labels.json'
    clips_dir = f'/content/clips/{sport_name}'
    
    if not os.path.exists(labels_file):
        print(f"{sport_name}: No labels, skipping")
        continue
    
    with open(labels_file) as f:
        sport_labels = json.load(f)
    
    cls_dir = f'/content/yolo_cls/{sport_name}'
    
    for clip_name, label in sport_labels.items():
        clip_path = f'{clips_dir}/{clip_name}'
        if not os.path.exists(clip_path):
            continue
        
        # Extract middle frame
        cap = cv2.VideoCapture(clip_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
        ret, frame = cap.read()
        cap.release()
        
        if not ret:
            continue
        
        frame = cv2.resize(frame, (640, 640))
        
        # 80/20 train/val split by hash
        split = 'train' if hash(clip_name) % 5 != 0 else 'val'
        out_dir = f'{cls_dir}/{split}/{label}'
        os.makedirs(out_dir, exist_ok=True)
        
        img_name = clip_name.replace('.mp4', '.jpg')
        cv2.imwrite(f'{out_dir}/{img_name}', frame)
    
    # Count
    train_count = sum(len(os.listdir(f'{cls_dir}/train/{d}')) 
                      for d in os.listdir(f'{cls_dir}/train') if os.path.isdir(f'{cls_dir}/train/{d}'))
    val_count = sum(len(os.listdir(f'{cls_dir}/val/{d}')) 
                    for d in os.listdir(f'{cls_dir}/val') if os.path.isdir(f'{cls_dir}/val/{d}'))
    print(f"{sport_name}: {train_count} train, {val_count} val frames")

## Section 6: YOLO Outcome Classifiers (Frame-Level)\nTrain YOLOv8 classification models for single-frame play type detection.\nThese complement VideoMAE by working on individual frames instead of video clips.

In [ ]:
#@title **5: Download VideoMAE Models** { display-mode: "form" }
for sport_name in sport_datasets:
    model_dir = f'{DRIVE_MODELS}/videomae_{sport_name}_v5'
    if os.path.exists(model_dir):
        # Zip and download
        zip_path = f'/content/videomae_{sport_name}_v5.zip'
        shutil.make_archive(zip_path.replace('.zip', ''), 'zip', model_dir)
        print(f"videomae_{sport_name}_v5: {os.path.getsize(zip_path)/1024/1024:.1f}MB")
        try:
            colab_files.download(zip_path)
        except:
            print(f"  Download from Drive: {model_dir}")
    else:
        print(f"videomae_{sport_name}_v5: Not trained yet")

print("\nAll VideoMAE models saved to Drive and downloaded!")

In [ ]:
#@title **4: Train VideoMAE Models (All Sports)** { display-mode: "form" }
from transformers import VideoMAEForVideoClassification, VideoMAEConfig
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

VIDEOMAE_EPOCHS = 20
VIDEOMAE_LR = 5e-5
VIDEOMAE_BATCH = 4

for sport_name, (train_ds, val_ds, label2id) in sport_datasets.items():
    model_name = f'videomae_{sport_name}_v5'
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    # Check for Drive checkpoint
    ckpt_path = f'{DRIVE_CHECKPOINTS}/{model_name}/model.pt'
    start_epoch = 0
    
    id2label = {v: k for k, v in label2id.items()}
    config = VideoMAEConfig.from_pretrained(
        'MCG-NJU/videomae-base',
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label,
    )
    model = VideoMAEForVideoClassification.from_pretrained(
        'MCG-NJU/videomae-base', config=config, ignore_mismatched_sizes=True
    ).cuda()
    
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt.get('epoch', 0)
        print(f"  Resumed from epoch {start_epoch}")
    
    optimizer = AdamW(model.parameters(), lr=VIDEOMAE_LR, weight_decay=0.05)
    scheduler = CosineAnnealingLR(optimizer, T_max=VIDEOMAE_EPOCHS)
    
    train_loader = DataLoader(train_ds, batch_size=VIDEOMAE_BATCH, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=VIDEOMAE_BATCH, num_workers=2)
    
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(start_epoch, VIDEOMAE_EPOCHS):
        # Train
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch in train_loader:
            pixel_values = batch['pixel_values'].cuda()
            labels_t = batch['labels'].cuda()
            
            outputs = model(pixel_values=pixel_values, labels=labels_t)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == labels_t).sum().item()
            total += labels_t.size(0)
        
        scheduler.step()
        train_acc = correct / max(total, 1)
        
        # Validate
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].cuda()
                labels_t = batch['labels'].cuda()
                outputs = model(pixel_values=pixel_values)
                preds = outputs.logits.argmax(dim=-1)
                val_correct += (preds == labels_t).sum().item()
                val_total += labels_t.size(0)
        
        val_acc = val_correct / max(val_total, 1)
        print(f"  Epoch {epoch+1}/{VIDEOMAE_EPOCHS} — loss: {total_loss/len(train_loader):.4f}, train_acc: {train_acc:.3f}, val_acc: {val_acc:.3f}")
        
        # Save best to Drive
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            save_dir = f'{DRIVE_CHECKPOINTS}/{model_name}'
            os.makedirs(save_dir, exist_ok=True)
            torch.save({
                'model_state': model.state_dict(),
                'epoch': epoch + 1,
                'val_acc': val_acc,
                'label2id': label2id,
            }, f'{save_dir}/model.pt')
            # Also save final model
            model.save_pretrained(f'{DRIVE_MODELS}/{model_name}')
            print(f"  [BEST] Saved — val_acc: {val_acc:.3f}")
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    
    print(f"  {model_name} done — best val_acc: {best_val_acc:.3f}")
    torch.cuda.empty_cache()

print("\n=== All VideoMAE models trained ===")

In [ ]:
#@title **3: Prepare VideoMAE Datasets** { display-mode: "form" }
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np

class ClipDataset(Dataset):
    """Load clips + labels for VideoMAE training."""
    def __init__(self, clips_dir, labels_dict, play_types, num_frames=16, img_size=224):
        self.num_frames = num_frames
        self.img_size = img_size
        self.play_types = play_types
        self.label2id = {t: i for i, t in enumerate(play_types)}
        
        self.samples = []
        for clip_path in sorted(glob.glob(f'{clips_dir}/*.mp4')):
            clip_name = os.path.basename(clip_path)
            if clip_name in labels_dict:
                label = labels_dict[clip_name]
                if label in self.label2id:
                    self.samples.append((clip_path, self.label2id[label]))
        
        print(f"  Dataset: {len(self.samples)} samples, {len(play_types)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Sample num_frames evenly
        indices = np.linspace(0, max(total - 1, 0), self.num_frames, dtype=int)
        frames = []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (self.img_size, self.img_size))
                frames.append(frame.astype(np.float32) / 255.0)
            else:
                frames.append(np.zeros((self.img_size, self.img_size, 3), dtype=np.float32))
        cap.release()
        
        # [T, H, W, C] -> [T, C, H, W]
        video = np.stack(frames)
        video = np.transpose(video, (0, 3, 1, 2))
        
        return {'pixel_values': torch.tensor(video), 'labels': torch.tensor(label)}

# Build datasets for available sports
sport_datasets = {}
for sport_name in ['basketball', 'football', 'lacrosse']:
    clips_dir = f'/content/clips/{sport_name}'
    labels_file = f'{DRIVE_DATA}/labels/{sport_name}_labels.json'
    
    if os.path.exists(labels_file):
        with open(labels_file) as f:
            sport_labels = json.load(f)
        
        if sport_labels:
            ds = ClipDataset(clips_dir, sport_labels, PLAY_TYPES[sport_name])
            if len(ds) >= 10:
                # 80/20 split
                train_size = int(0.8 * len(ds))
                val_size = len(ds) - train_size
                train_ds, val_ds = torch.utils.data.random_split(ds, [train_size, val_size])
                sport_datasets[sport_name] = (train_ds, val_ds, ds.label2id)
                print(f"  {sport_name}: {train_size} train, {val_size} val")
            else:
                print(f"  {sport_name}: Only {len(ds)} samples — need at least 10, skipping")
    else:
        print(f"  {sport_name}: No labels found — run Section 2 first")

print(f"\nReady to train: {list(sport_datasets.keys())}")

## Section 3-5: VideoMAE Play Classification (All 3 Sports)\nFine-tune VideoMAE (MCG-NJU/videomae-base) for temporal play classification.\nEach sport gets its own model. Drive backup after every epoch.

In [ ]:
#@title **2A: Auto-Label Clips with Claude Vision** { display-mode: "form" }
import anthropic
import base64

PLAY_TYPES = {
    'basketball': ['layup', 'jump_shot', 'dunk', 'three_pointer', 'fast_break', 
                    'rebound', 'steal', 'block', 'assist', 'free_throw', 'turnover', 'other'],
    'football': ['pass_play', 'run_play', 'touchdown', 'interception', 'sack',
                 'field_goal', 'punt', 'kickoff', 'tackle', 'catch', 'other'],
    'lacrosse': ['shot', 'goal', 'save', 'ground_ball', 'face_off',
                 'clear', 'dodge', 'pass', 'ride', 'other']
}

# Load existing labels from Drive (never re-label paid clips)
LABELS_FILE = f'{DRIVE_DATA}/labels/{SPORT}_labels.json'
os.makedirs(f'{DRIVE_DATA}/labels', exist_ok=True)
labels = {}
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE) as f:
        labels = json.load(f)
    print(f"Loaded {len(labels)} existing labels from Drive")

clips = sorted(glob.glob(f'{CLIPS_DIR}/*.mp4'))
unlabeled = [c for c in clips if os.path.basename(c) not in labels]
print(f"Total clips: {len(clips)}, Already labeled: {len(labels)}, To label: {len(unlabeled)}")

if not ANTHROPIC_API_KEY:
    print("\nERROR: Set ANTHROPIC_API_KEY in Section 0 first!")
elif unlabeled:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    types_list = ', '.join(PLAY_TYPES[SPORT])
    session_cost = 0
    
    for i, clip_path in enumerate(unlabeled):
        clip_name = os.path.basename(clip_path)
        print(f"\r  Labeling {i+1}/{len(unlabeled)}: {clip_name}", end='')
        
        # Extract 3 frames (start, middle, end) for classification
        cap = cv2.VideoCapture(clip_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames_b64 = []
        
        for frac in [0.1, 0.5, 0.9]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(total_frames * frac))
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (512, 288))
                _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
                frames_b64.append(base64.b64encode(buf).decode())
        cap.release()
        
        if len(frames_b64) < 2:
            labels[clip_name] = 'other'
            continue
        
        # Build Claude message with frames
        content = []
        for fb64 in frames_b64:
            content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": fb64}})
        content.append({"type": "text", "text": f"This is a {SPORT} clip. Classify into EXACTLY one: {types_list}. Reply with ONLY the label, nothing else."})
        
        try:
            resp = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=20,
                messages=[{"role": "user", "content": content}]
            )
            label = resp.content[0].text.strip().lower().replace(' ', '_')
            if label not in PLAY_TYPES[SPORT]:
                label = 'other'
            labels[clip_name] = label
            
            # Cost tracking (~$0.025/clip for 3 frames)
            cost = (resp.usage.input_tokens * 0.25 + resp.usage.output_tokens * 1.25) / 1_000_000
            session_cost += cost
            
        except Exception as e:
            print(f"\n  Error on {clip_name}: {e}")
            labels[clip_name] = 'other'
        
        # Save to Drive every 10 clips
        if (i + 1) % 10 == 0:
            with open(LABELS_FILE, 'w') as f:
                json.dump(labels, f, indent=2)
    
    # Final save
    with open(LABELS_FILE, 'w') as f:
        json.dump(labels, f, indent=2)
    track_cost('claude_labeling', session_cost)
    print(f"\n\nDone! Labeled {len(unlabeled)} clips. Session cost: ${session_cost:.4f}")

# Show label distribution
from collections import Counter
dist = Counter(labels.values())
print("\nLabel distribution:")
for label, count in dist.most_common():
    print(f"  {label}: {count}")

## Section 2: Claude Vision Auto-Labeling\nUse Claude Vision to classify each clip into play types. Cost: ~$0.025/clip.\nLabels are saved to Drive so you never pay twice for the same clip.

In [ ]:
#@title **1B: Split Videos into Clips (Scene Detection)** { display-mode: "form" }
from scenedetect import detect, ContentDetector
import cv2

CLIPS_DIR = f'/content/clips/{SPORT}'
os.makedirs(CLIPS_DIR, exist_ok=True)

MIN_CLIP_SEC = 2
MAX_CLIP_SEC = 15

total_clips = 0
for video_path in glob.glob(f'{DOWNLOAD_DIR}/*.mp4'):
    vid_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f"\nProcessing: {vid_name}")
    
    try:
        scenes = detect(video_path, ContentDetector(threshold=27.0))
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        
        clip_num = 0
        for start, end in scenes:
            duration = (end.get_frames() - start.get_frames()) / fps
            if MIN_CLIP_SEC <= duration <= MAX_CLIP_SEC:
                clip_path = f'{CLIPS_DIR}/{vid_name}_clip{clip_num:04d}.mp4'
                
                # Extract clip using ffmpeg (faster than OpenCV)
                start_sec = start.get_frames() / fps
                subprocess.run([
                    'ffmpeg', '-y', '-ss', str(start_sec), '-i', video_path,
                    '-t', str(duration), '-c:v', 'libx264', '-preset', 'fast',
                    '-crf', '23', '-an', clip_path
                ], capture_output=True)
                
                if os.path.exists(clip_path) and os.path.getsize(clip_path) > 10000:
                    clip_num += 1
                    total_clips += 1
        
        cap.release()
        print(f"  Extracted {clip_num} clips")
    except Exception as e:
        print(f"  Error: {e}")

# Backup clips to Drive
drive_clips = f'{DRIVE_DATA}/clips/{SPORT}'
os.makedirs(drive_clips, exist_ok=True)
for c in glob.glob(f'{CLIPS_DIR}/*.mp4'):
    dest = f'{drive_clips}/{os.path.basename(c)}'
    if not os.path.exists(dest):
        shutil.copy2(c, dest)

print(f"\n=== Total clips: {total_clips} ===")
print(f"Backed up to Drive: {drive_clips}")

In [ ]:
#@title **1A: Download YouTube Videos** { display-mode: "form" }
import yt_dlp

# Add your YouTube URLs here (game film, highlight reels, etc.)
YOUTUBE_URLS = [
    # "https://www.youtube.com/watch?v=EXAMPLE1",
    # "https://www.youtube.com/watch?v=EXAMPLE2",
]

SPORT = "basketball"  #@param ["basketball", "football", "lacrosse"]
DOWNLOAD_DIR = f'/content/raw_videos/{SPORT}'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Also check Drive for previously downloaded videos
drive_vids = f'{DRIVE_DATA}/raw_videos/{SPORT}'
os.makedirs(drive_vids, exist_ok=True)
existing = glob.glob(f'{drive_vids}/*.mp4')
if existing:
    print(f"Found {len(existing)} videos on Drive, copying to local...")
    for v in existing:
        dest = f'{DOWNLOAD_DIR}/{os.path.basename(v)}'
        if not os.path.exists(dest):
            shutil.copy2(v, dest)

if YOUTUBE_URLS:
    ydl_opts = {
        'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]',
        'outtmpl': f'{DOWNLOAD_DIR}/%(id)s.%(ext)s',
        'merge_output_format': 'mp4',
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download(YOUTUBE_URLS)
    
    # Backup to Drive
    for f in glob.glob(f'{DOWNLOAD_DIR}/*.mp4'):
        shutil.copy2(f, f'{drive_vids}/{os.path.basename(f)}')
    print("Videos backed up to Drive")

videos = glob.glob(f'{DOWNLOAD_DIR}/*.mp4')
print(f"\nTotal videos ready: {len(videos)}")
if not videos:
    print("Add YouTube URLs above or upload videos to:", DOWNLOAD_DIR)

## Section 1: YouTube Download + Scene Detection\nDownload game film from YouTube URLs and split into 2-15 second clips using PySceneDetect.

In [ ]:
#@title **Helper Functions — Crash Protection + Training** { display-mode: "form" }
from ultralytics import YOLO
from google.colab import files as colab_files

def make_drive_backup_callback(model_name):
    """Returns YOLO callback that backs up to Drive every 5 epochs."""
    backup_dir = f'{DRIVE_CHECKPOINTS}/{model_name}'
    os.makedirs(backup_dir, exist_ok=True)
    
    def on_train_epoch_end(trainer):
        epoch = trainer.epoch + 1
        if epoch % 5 == 0:
            try:
                for fname in ['best.pt', 'last.pt']:
                    src = os.path.join(trainer.save_dir, 'weights', fname)
                    if os.path.exists(src):
                        shutil.copy2(src, f'{backup_dir}/{fname}')
                print(f"  [Drive Backup] {model_name} epoch {epoch} saved")
            except Exception as e:
                print(f"  [Drive Backup] Warning: {e}")
    
    return on_train_epoch_end

def find_resume_checkpoint(model_name):
    """Check Drive for a checkpoint to resume from."""
    backup_dir = f'{DRIVE_CHECKPOINTS}/{model_name}'
    last_pt = f'{backup_dir}/last.pt'
    if os.path.exists(last_pt):
        size_mb = os.path.getsize(last_pt) / 1024 / 1024
        if size_mb > 1:  # Sanity check — must be > 1MB
            print(f"  [Resume] Found checkpoint for {model_name} ({size_mb:.1f}MB)")
            return last_pt
    return None

def safe_train(model_or_path, data, epochs, imgsz, batch, model_name, 
               project='/content/runs', extra_args=None):
    """Train with auto-resume, Drive backup, and OOM retry."""
    extra_args = extra_args or {}
    
    # Check for resume checkpoint
    resume_pt = find_resume_checkpoint(model_name)
    if resume_pt:
        print(f"  Resuming {model_name} from Drive checkpoint...")
        model = YOLO(resume_pt)
        extra_args['resume'] = True
    elif isinstance(model_or_path, str):
        model = YOLO(model_or_path)
    else:
        model = model_or_path
    
    # Add Drive backup callback
    model.add_callback('on_train_epoch_end', make_drive_backup_callback(model_name))
    
    try:
        results = model.train(
            data=data, epochs=epochs, imgsz=imgsz, batch=batch,
            project=project, name=model_name,
            save_period=5, amp=True, cache=True,
            patience=25 if epochs >= 100 else 15,
            **extra_args
        )
        return model, results
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f"  [OOM] Retrying {model_name} with batch={batch//2}...")
            import torch; torch.cuda.empty_cache()
            results = model.train(
                data=data, epochs=epochs, imgsz=imgsz, batch=batch//2,
                project=project, name=model_name,
                save_period=5, amp=True, cache=True,
                patience=25 if epochs >= 100 else 15,
                **extra_args
            )
            return model, results
        raise

def download_model(model_name, src_dir=None):
    """Copy best.pt to Drive and trigger browser download."""
    if src_dir is None:
        src_dir = f'/content/runs/{model_name}/weights'
    
    best_pt = f'{src_dir}/best.pt'
    if not os.path.exists(best_pt):
        # Try finding it
        found = glob.glob(f'/content/runs/**/{model_name}*/weights/best.pt', recursive=True)
        if found:
            best_pt = found[0]
        else:
            print(f"ERROR: No best.pt found for {model_name}")
            return
    
    size_mb = os.path.getsize(best_pt) / 1024 / 1024
    if size_mb < 1:
        print(f"WARNING: {model_name} best.pt is only {size_mb:.2f}MB — may be corrupted")
    
    # Copy to Drive
    drive_dest = f'{DRIVE_MODELS}/{model_name}_best.pt'
    shutil.copy2(best_pt, drive_dest)
    print(f"Saved to Drive: {drive_dest} ({size_mb:.1f}MB)")
    
    # Browser download
    try:
        colab_files.download(best_pt)
        print(f"Browser download triggered for {model_name}")
    except Exception as e:
        print(f"Browser download failed (headless?): {e}")
        print(f"Download manually from Drive: {drive_dest}")

def track_cost(model_name, cost_usd):
    """Persistent API cost tracker on Drive."""
    costs = {}
    if os.path.exists(DRIVE_COSTS):
        with open(DRIVE_COSTS) as f:
            costs = json.load(f)
    costs[model_name] = costs.get(model_name, 0) + cost_usd
    costs['total'] = sum(v for k, v in costs.items() if k != 'total')
    with open(DRIVE_COSTS, 'w') as f:
        json.dump(costs, f, indent=2)
    print(f"  Cost: ${cost_usd:.4f} | Total: ${costs['total']:.4f}")

print("Helper functions loaded!")

In [ ]:
#@title **Save API Keys to Drive** (run once) { display-mode: "form" }
# Fill in your keys in the cell above, then run this to persist them
if ANTHROPIC_API_KEY or ROBOFLOW_API_KEY:
    with open(f'{DRIVE_ROOT}/api_keys.json', 'w') as f:
        json.dump({'anthropic': ANTHROPIC_API_KEY, 'roboflow': ROBOFLOW_API_KEY}, f)
    print("Keys saved to Drive — will auto-load on reconnect")
else:
    print("No keys set yet. Fill them in the Setup cell first.")

In [ ]:
#@title **Section 0: Setup — Run This First** { display-mode: "form" }
# Mount Google Drive for crash protection
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, glob

# === DRIVE BACKUP PATHS ===
DRIVE_ROOT = '/content/drive/MyDrive/clipt_v5_training'
DRIVE_MODELS = f'{DRIVE_ROOT}/models'
DRIVE_DATA = f'{DRIVE_ROOT}/datasets'
DRIVE_COSTS = f'{DRIVE_ROOT}/api_costs.json'
DRIVE_CHECKPOINTS = f'{DRIVE_ROOT}/checkpoints'

for d in [DRIVE_MODELS, DRIVE_DATA, DRIVE_CHECKPOINTS]:
    os.makedirs(d, exist_ok=True)

# === GPU CHECK ===
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                         capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu_info}")
HAS_A100 = 'A100' in gpu_info
DEFAULT_BATCH = 16 if HAS_A100 else 8
print(f"Default batch size: {DEFAULT_BATCH}")

# === INSTALL DEPS ===
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'roboflow', 'transformers', 
                'datasets', 'accelerate', 'scenedetect[opencv]', 'yt-dlp',
                'anthropic', 'Pillow', 'tqdm'], check=True)
print("All dependencies installed!")

# === API KEYS (set yours) ===
ANTHROPIC_API_KEY = ""  # For Claude Vision auto-labeling
ROBOFLOW_API_KEY = ""   # For dataset downloads

# Load from Drive if saved
key_file = f'{DRIVE_ROOT}/api_keys.json'
if os.path.exists(key_file):
    with open(key_file) as f:
        keys = json.load(f)
        ANTHROPIC_API_KEY = keys.get('anthropic', ANTHROPIC_API_KEY)
        ROBOFLOW_API_KEY = keys.get('roboflow', ROBOFLOW_API_KEY)
    print("Loaded API keys from Drive")
else:
    print("Set your API keys above, then run the save cell below")

print("\n=== SETUP COMPLETE ===")

# V5 Auto-Label Training Pipeline
**10 Models | Crash Protection | Autopilot Ready**

## Models:
1. VideoMAE Basketball
2. VideoMAE Football
3. VideoMAE Lacrosse
4. YOLO Outcome Classifier (3 sports)
5. Jersey OCR v5 (FIXED: 10 digit classes)
6. Player Detector v5 (FIXED: Roboflow datasets)
7. Ball Detector v5 (NEW)
8. Referee Detector v5 (NEW)

## Crash Protection:
- Google Drive backup every 5 epochs
- Auto-resume from last.pt on reconnect
- Persistent API cost tracker on Drive
- Download cell after every model